### Code to extract documents for all chosen source domains in preparation for metaphor analysis

Chosen source domains:
- Cognitive Agency:
    - trainee -> Auszubildender
    - god -> Gott
- Behavioral Agency:
    - conversational partner: child, friend -> Kind, Freund
- Labor:
    - artist -> Künstler
    - assistant -> Assistent
    - doctor -> Arzt
    - (expert -> Experte)
    - teacher -> Lehrer
    - thief -> Dieb
    - programmer -> Programmierer
    - writer -> Author
    - (co)worker -> Kollege (Mitarbeiter)

In [19]:
from pprint import pprint
import os
import re
import json
import ast
import pandas as pd
import numpy as np
import string
import tqdm
from nltk import FreqDist

import gensim
from gensim.models import LdaModel
from sklearn.feature_extraction.text import TfidfVectorizer

import spacy
nlp = spacy.load('de_core_news_lg')

def flatten(xss):
    return [x for xs in xss for x in xs]

plots_dir = "plots/"

In [20]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out

# search with stemm/lemma
def get_lemma(sent):
    doc = nlp(sent)
    return [token.lemma_ for token in doc]

In [21]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
# stopwords_l a list of stopwords in the target language if not english
def calc_tf_idf(corpus, stopwords_l= None):
    corpus_cleaned = [remove_punctuation(doc.lower()) for doc in corpus]
    if stopwords_l:
        corpus_cleaned = [remove_stop_words_from_text(doc.split()) for doc in corpus_cleaned]
        vectorizer = TfidfVectorizer(stop_words=stopwords_l, lowercase=True, ngram_range=(1,1))
    else:
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1,1))
    # fit tfidf
    tfidf_m = vectorizer.fit_transform(corpus_cleaned) #transform
    #print(tfidf_m.shape)
    feature_array = vectorizer.get_feature_names_out()
    # top (20?) for every doc
    tfidf_sorting = np.argsort(tfidf_m.toarray()).flatten()[::-1] 
    top_n = feature_array[tfidf_sorting].tolist()[:30]
    #print(top_n)
    return top_n #array of list(s?)

In [22]:
# list of domains
source_doms = ['Auszubildender', 'Gott', 'Kind', 'Freund', 'Künstler', 'Assistent', 'Arzt', 'Lehrer', 'Dieb', 'Programmierer', 'Author', 'Kollege']
source_doms_lemma = [get_lemma(x)[0].lower() for x in source_doms]
print(len(source_doms_lemma))
print(source_doms_lemma)

12
['auszubildende', 'gott', 'kind', 'freund', 'künstler', 'assistent', 'arzt', 'lehrer', 'dieb', 'programmierer', 'author', 'kollege']


#### RegEx search for sentences 

In [23]:
# build lexcons for each domain, label some examples per domain using regex search
# get sents in Leipzig data, sort into doms

def get_sentences_doms(leipzig_main,source_doms_lemma, stopwords):
    sentences = {}
    sentences_doms = pd.DataFrame(columns=['id', 'sentence', 'domain'])
    sentences_doms_re = pd.DataFrame(columns=['id', 'sentence', 'domain'])

    d_counter = [0] * len(source_doms_lemma)
    d_counter_re = [0] * len(source_doms_lemma)
    for p in os.listdir(leipzig_main):
        print(p)
        if (x < 1000 for x in d_counter) and (x < 1000 for x in d_counter_re):
            sentences_file = os.path.join(leipzig_main, p, f"{p}-sentences.txt")
            # extract sentences
            sentences = open(sentences_file).readlines()
            n=1
            for chunk in tqdm.tqdm([sentences[i:i+n] for i in range(0, len(sentences), n)]):
                try:
                    topn = calc_tf_idf(chunk, stopwords_l=stopwords)
                except:
                    print("error in tfidf")
                    topn = []
                for d in source_doms_lemma:
                    if d in topn and d_counter[source_doms_lemma.index(d)] < 1000:
                        sentences_doms.loc[len(sentences_doms)] = [len(sentences_doms), chunk, d]
                        d_counter[source_doms_lemma.index(d)] += 1
                
                    if d in [get_lemma(x.lower())[0] for x in chunk] and d_counter_re[source_doms_lemma.index(d)] < 1000:
                        sentences_doms_re.loc[len(sentences_doms_re)] = [len(sentences_doms_re), chunk, d]
                        d_counter_re[source_doms_lemma.index(d)] += 1
    return sentences_doms, sentences_doms_re

In [ ]:
leipzig_main = "deu_news"
get_sentences_doms(leipzig_main, source_doms_lemma, stopwords)

deu_news_2003_1M


  8%|▊         | 81697/1000000 [21:57:37<22:30:47, 11.33it/s]     

#### Cluster sentences for each domain

In [ ]:
[s.split() for s in source_doms_lemma]

In [ ]:
# set values and create list of processed words here souce_doms and source_doms_lemma

# create bag of words for lda model
dictionary = gensim.corpora.Dictionary([s.split() for s in source_doms_lemma])
corpus = [dictionary.doc2bow(value) for value in [s.split() for s in source_doms_lemma]]

# train the lda model
lda_model = gensim.models.ldamodel.LdaModel(
    corpus=corpus, 
    num_topics=len(source_doms_lemma), 
    id2word=dictionary, 
    passes=100,
    eval_every = 1)
lda_model.save('lda_train.model')
lda_topics = lda_model.print_topics()

# print the topics to screen
for t in lda_topics:
    print(f"Topic {t[0]+1}: {t[1]}")

In [ ]:
# classify domains based on the ten LDA topics 

# create bag of words from moral values list and classify topics according to values list
bow_vector = dictionary.doc2bow(source_doms)
for index, score in sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1]):
    print("Score: {}\t Topic: {}".format(score, lda_model.print_topic(index, 1)))

# create frequency distribution graph from the moral values table
# create freq dist list and graph from moral values list
dist = FreqDist(source_doms)
dist_values = dist.most_common(10)
for v in dist_values:
    print(f"{v[0]}: {v[1]}")

dist.plot(10, title="Frequency Distribution of Source Domains")


In [ ]:
'''
# use to cluster, save files
source_d_path = "source_domains"
dom_sentences = pd.DataFrame(columns=['id','domain','sentence'])
for p in source_doms_lemma:
    cluster_results = {}
    for domain_lemma in sentences_all['sentence']:
        if p in domain_lemma:
            cluster_results[sentences_all.loc[sentences_all['sentence'] == domain_lemma, 'id'].values[0]] = domain_lemma

    # to df
    s_id = 0
    sentence = ''
    dom_sentences.loc[len(dom_sentences)] = [s_id, p, sentence]
    # save to file
    dom_sentences.to_csv(os.path.join(source_d_path, p, f"{p}_sentences.txt"))
'''